In [ ]:
# # DenseNet121 Model Training

# This notebook trains DenseNet121 for multi-label thoracic disease classification.

# Tasks:
# - Model architecture definition
# - Loss function setup
# - Training loop
# - Validation pipeline
# - Model checkpoint generation

In [ ]:
import os
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms
from PIL import Image
import pandas as pd
from tqdm import tqdm


In [ ]:
# Paths
IMAGE_DIR = "/content/drive/MyDrive/DS3 (2)/archive/images/images_normalized"
CSV_PATH  = "/content/drive/MyDrive/final_training_labels_LOCKED.csv"

# Training params
BATCH_SIZE = 16
EPOCHS = 15
LR = 1e-4
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"


In [ ]:
class CXRDataset(Dataset):
    def __init__(self, csv_path, image_dir, transform=None):
        self.df = pd.read_csv(csv_path)
        self.image_dir = image_dir
        self.transform = transform

        self.label_cols = [
            "lung_opacity",
            "consolidation",
            "pleural_effusion",
            "cardiomegaly",
            "atelectasis",
            "edema",
            "support_devices"
        ]

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        img_path = os.path.join(self.image_dir, row["filename"])
        image = Image.open(img_path).convert("RGB")

        labels = torch.tensor(
        row[self.label_cols].astype(float).values,
        dtype=torch.float32)



        if self.transform:
            image = self.transform(image)

        return image, labels


In [ ]:
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])


In [ ]:
dataset = CXRDataset(CSV_PATH, IMAGE_DIR, train_transform)
dataloader = DataLoader(
    dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=2,
    pin_memory=True
)

print("Total training samples:", len(dataset))


Total training samples: 3002


In [ ]:
model = models.densenet121(pretrained=True)

num_features = model.classifier.in_features
model.classifier = nn.Linear(num_features, 7)

model = model.to(DEVICE)


/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=DenseNet121_Weights.IMAGENET1K_V1`. You can also use `weights=DenseNet121_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Downloading: "https://download.pytorch.org/models/densenet121-a639ec97.pth" to /root/.cache/torch/hub/checkpoints/densenet121-a639ec97.pth


100%|██████████| 30.8M/30.8M [00:00<00:00, 155MB/s]


In [ ]:
#light fine tuning
for param in model.parameters():
    param.requires_grad = False

# Unfreeze classifier + last dense block
for param in model.classifier.parameters():
    param.requires_grad = True

for param in model.features.denseblock4.parameters():
    param.requires_grad = True


In [ ]:
criterion = nn.BCEWithLogitsLoss()

optimizer = torch.optim.Adam(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=LR
)


In [ ]:
CHECKPOINT_PATH = "/content/drive/MyDrive/densenet_cxr_checkpoint.pth"


In [ ]:
start_epoch = 0

if os.path.exists(CHECKPOINT_PATH):
    checkpoint = torch.load(CHECKPOINT_PATH, map_location=DEVICE)
    model.load_state_dict(checkpoint["model_state"])
    optimizer.load_state_dict(checkpoint["optimizer_state"])
    start_epoch = checkpoint["epoch"] + 1

    print(f"✅ Resuming training from epoch {start_epoch}")
else:
    print("🆕 No checkpoint found. Starting fresh training.")


🆕 No checkpoint found. Starting fresh training.


In [ ]:
import pandas as pd

CSV_PATH = "/content/drive/MyDrive/final_training_labels.csv"

df = pd.read_csv(CSV_PATH)

label_cols = [
    "lung_opacity",
    "consolidation",
    "pleural_effusion",
    "cardiomegaly",
    "atelectasis",
    "edema",
    "support_devices"
]

# Force numeric + handle bad values
for col in label_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce").fillna(0).astype(float)

# Save cleaned CSV
CLEAN_CSV_PATH = "/content/drive/MyDrive/final_training_labels_clean.csv"
df.to_csv(CLEAN_CSV_PATH, index=False)

print("✅ Cleaned labels saved to:", CLEAN_CSV_PATH)
print(df[label_cols].dtypes)


✅ Cleaned labels saved to: /content/drive/MyDrive/final_training_labels_clean.csv
lung_opacity        float64
consolidation       float64
pleural_effusion    float64
cardiomegaly        float64
atelectasis         float64
edema               float64
support_devices     float64
dtype: object


In [ ]:
#main training loop
model.train()

for epoch in range(start_epoch, EPOCHS):
    running_loss = 0.0

    for images, labels in tqdm(dataloader, desc=f"Epoch {epoch+1}/{EPOCHS}"):
        images = images.to(DEVICE)
        labels = labels.to(DEVICE)

        optimizer.zero_grad()

        outputs = model(images)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    avg_loss = running_loss / len(dataloader)
    print(f"Epoch {epoch+1} completed | Avg Loss: {avg_loss:.4f}")

    # 🔒 SAVE CHECKPOINT AFTER EACH EPOCH
    torch.save({
        "epoch": epoch,
        "model_state": model.state_dict(),
        "optimizer_state": optimizer.state_dict(),
        "loss": avg_loss
    }, CHECKPOINT_PATH)

    print(f"💾 Checkpoint saved at epoch {epoch+1}")


Epoch 1/15: 100%|██████████| 188/188 [12:02<00:00,  3.84s/it]


Epoch 1 completed | Avg Loss: 0.4479
💾 Checkpoint saved at epoch 1


Epoch 2/15: 100%|██████████| 188/188 [03:14<00:00,  1.03s/it]


Epoch 2 completed | Avg Loss: 0.3822
💾 Checkpoint saved at epoch 2


Epoch 3/15: 100%|██████████| 188/188 [03:11<00:00,  1.02s/it]


...

Epoch 4 completed | Avg Loss: 0.3349
💾 Checkpoint saved at epoch 4


Epoch 5/15: 100%|██████████| 188/188 [03:09<00:00,  1.01s/it]


Epoch 5 completed | Avg Loss: 0.3036
💾 Checkpoint saved at epoch 5


Epoch 6/15:  86%|████████▌ | 161/188 [02:43<00:27,  1.01s/it]


KeyboardInterrupt: 

In [ ]:
#inference code

In [ ]:
import torch
import torchvision.models as models
import torch.nn as nn

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# Rebuild the model architecture (must match training)
model = models.densenet121(pretrained=False)
model.classifier = nn.Linear(model.classifier.in_features, 7)

# Load checkpoint
checkpoint = torch.load(
    "/content/drive/MyDrive/densenet_cxr_checkpoint.pth",
    map_location=DEVICE
)

model.load_state_dict(checkpoint["model_state"])
model = model.to(DEVICE)
model.eval()

print("✅ Model loaded and ready for inference")


/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=None`.
  warnings.warn(msg)


✅ Model loaded and ready for inference


In [ ]:
LABELS = [
    "lung_opacity",
    "consolidation",
    "pleural_effusion",
    "cardiomegaly",
    "atelectasis",
    "edema",
    "support_devices"
]


In [ ]:
from torchvision import transforms
from PIL import Image

infer_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])


In [ ]:
img_path = "/content/drive/MyDrive/DS3 (2)/100_images/1226_IM-0150-1001.dcm.png"

img = Image.open(img_path).convert("RGB")
img = infer_transform(img).unsqueeze(0).to(DEVICE)

with torch.no_grad():
    logits = model(img)
    probs = torch.sigmoid(logits).cpu().numpy()[0]

for label, p in zip(LABELS, probs):
    print(f"{label:20s}: {p:.3f}")


lung_opacity        : 0.176
consolidation       : 0.533
pleural_effusion    : 0.709
cardiomegaly        : 0.015
atelectasis         : 0.028
edema               : 0.093
support_devices     : 0.152
